# Backpropagation

_Deep Learning_

**Run the chain rule backward to find how each weight caused the error, then fix it.**

Backpropagation is how a network learns.

     Forward propagation gives a prediction and a loss. Backprop works backward to find how much each weight is to blame for that loss.

---

This notebook builds backprop up one step at a time: a single neuron's forward pass, the backward pass that fills in the gradients, then a real digit network so you can see which layers actually move. Run each cell and read the note above it. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We'll take a single neuron, push one input through it (the **forward pass**), then run backprop (the **backward pass**) to get the gradient of the loss with respect to every weight, and finally take one optimizer step.

### Step 1 — Build the neuron, optimizer, and loss

`nn.Linear(3, 1)` is one neuron with three input weights and a bias. We pair it with plain **SGD** (the rule `w ← w - lr·grad`) and **mean-squared-error** loss. None of this has touched data yet — we're just wiring up the pieces backprop will use.

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(3, 1)                          # one neuron: 3 weights + bias
opt = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

### Step 2 — Forward pass: predict, then measure the error

We feed in one input vector and ask for a target. The model produces a prediction (`pred`), and the loss function measures how far that prediction is from the target. This forward pass also quietly records the computation graph PyTorch will walk backward in the next step.

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0]])              # one input example
target = torch.tensor([[10.0]])                  # the value we want the neuron to output

pred = model(x)                                  # forward pass: run the input through the neuron
loss = loss_fn(pred, target)                     # measure how wrong the prediction is

### Step 3 — Backward pass: chain the rule back to every weight

`loss.backward()` is backpropagation itself: it applies the chain rule from the loss back through the graph, filling each weight's `.grad` with `dL/dw` — how much that weight contributed to the error. Then `opt.step()` nudges every weight downhill by `lr · grad`. We clear stale gradients first, because PyTorch accumulates them by default.

In [ ]:
opt.zero_grad()                                  # clear any gradients left from before
loss.backward()                                  # chain rule: fill each weight's .grad
print("dL/dw:", model.weight.grad)               # the gradient for every weight
opt.step()                                       # w <- w - lr * grad
print("loss:", round(loss.item(), 4))

## Visualize the data & results

_When real gradients flow back through a trained digit network, which layers get the biggest weight updates?_

### Step 1 — Train one backward step on real digits

We load the 8x8 digit images, scale the pixels into `[0, 1]`, and build a small four-layer network. `warm_start=True` with `max_iter=1` lets us run exactly **one** backward pass at a time, so we can snapshot the weights and watch a single SGD step in isolation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier

digits = load_digits()
X = digits.data / 16.0          # scale pixel intensities (0-16) into [0, 1]
y = digits.target

net = MLPClassifier(hidden_layer_sizes=(32, 16, 8), max_iter=1, warm_start=True,
                    random_state=1, alpha=1e-3)
net.partial_fit(X, y, classes=np.unique(y))   # one backward pass

### Step 2 — Snapshot the weights, then take one more step

We copy every layer's weight matrix, run **one more** backward pass, and measure how much each layer changed (`deltas` = mean absolute weight update per layer). This is the empirical size of the gradient signal reaching each layer in a single SGD step.

In [ ]:
before = [w.copy() for w in net.coefs_]       # snapshot weights of every layer
net.partial_fit(X, y)                          # one more SGD step
deltas = [np.abs(a - b).mean() for a, b in zip(net.coefs_, before)]   # mean update per layer

### Step 3 — Plot the per-layer update size

We order the layers output-first to mirror the direction backprop travels. The bar heights show how much each layer moved. Layers nearer the output typically see a larger, less-diluted gradient, while the input layer's update is smaller — the gradient has been multiplied down on its way back.

In [ ]:
labels = ["output layer", "hidden 2", "hidden 1", "input layer"]
values = deltas[::-1]                           # reverse so the output layer comes first
colors = ["#7ee787", "#4ea1ff", "#4ea1ff", "#ff7b72"]

plt.bar(labels, values, color=colors)
plt.title("Real mean weight-update per layer, digit MLP (one SGD step)")
plt.xticks(rotation=15)
plt.show()